# Week 3 실습: 연결 리스트 & ADT

**데이터구조 | 3주차**  
이 노트북에서는 리스트 ADT를 명세하고, 단순 연결 리스트를 Python으로 구현하며, 삽입·삭제·탐색·순회 연산의 연결 변화를 확인합니다.

## 학습목표
1. ADT가 정의하는 **무엇(인터페이스)** 과 자료구조/구현이 결정하는 **어떻게** 를 구분한다.
2. 배열 기반 리스트와 연결 기반 리스트의 접근·삽입·삭제 비용을 비교한다.
3. `Node`, `LinkedList`를 이용해 단순 연결 리스트의 핵심 연산을 구현하고 검증한다.
4. 링크 재연결만으로 리스트를 뒤집고, 두 포인터로 중간 노드를 찾는다.

In [1]:
# 실습 환경 확인
import sys
print('Python', sys.version.split()[0])

Python 3.13.15


---
## 1. ADT: 무엇을 제공하는가?

**리스트 ADT**는 순서가 있는 원소들의 유한한 모음과 다음 연산을 명세합니다.

|연산|의미|
|---|---|
|`insert(i, x)`|위치 `i`에 `x` 삽입|
|`delete(i)`|위치 `i`의 원소 제거 및 반환|
|`get(i)`|위치 `i`의 원소 조회|
|`search(x)`|`x`의 위치 반환(없으면 -1)|
|`length()` / `is_empty()`|크기 / 공백 여부|

ADT는 배열인지 연결 리스트인지는 말하지 않습니다. 그것은 **구현 선택**입니다.

### 생각해 보기
- Stack ADT의 `push`, `pop`, `peek`은 구현 방식과 무관하게 어떤 약속을 해야 할까요?
- Python의 `list`는 배열 기반 구현이지만, 우리가 만들 `LinkedList`도 같은 리스트 ADT의 일부 연산을 제공할 수 있습니다.

---
## 2. 배열 기반 vs 연결 기반

|연산/특성|배열 기반 리스트|단순 연결 리스트|
|---|---:|---:|
|인덱스로 i번째 원소 접근|$O(1)$|$O(n)$|
|맨 앞 삽입/삭제|$O(n)$|$O(1)$|
|중간 삽입/삭제|$O(n)$|이전 노드를 알고 있으면 $O(1)$|
|크기 변화|재할당 가능|노드 단위로 동적 증가|
|추가 메모리|작음|노드마다 링크 필요|

> 주의: 연결 리스트의 `insert_last`는 **tail을 저장하지 않으면** 마지막 노드까지 순회해야 하므로 $O(n)$입니다.

---
## 실습 규칙
- 아래의 `TODO`를 순서대로 완성하세요.
- 제공된 테스트를 통과하도록 구현합니다.
- 오류가 나면 `print()`로 `head`, 현재 노드, `link`의 변화를 확인하세요.
- **중요:** 삽입 시 기존 연결을 잃지 않도록 연결 순서를 지키세요.

## 3. 구현 골격

In [2]:
from dataclasses import dataclass
from typing import Any, Optional, Iterator


@dataclass
class Node:
    data: Any
    link: Optional["Node"] = None


class LinkedList:
    def __init__(self):
        self.head: Optional[Node] = None
        self.size = 0

    def is_empty(self) -> bool:
        # TODO 1: 빈 리스트인지 반환하세요.
        return self.head is None

    def __len__(self) -> int:
        # TODO 2: 원소 수를 반환하세요.
        return self.size

    def __iter__(self) -> Iterator[Any]:
        # TODO 3: head부터 link를 따라가며 data를 하나씩 yield하세요.
        current = self.head

        while current is not None:
            yield current.data
            current = current.link

    def to_list(self):
        return list(self)

    def insert_first(self, data: Any) -> None:
        # TODO 4: new_node의 link를 기존 head에 연결한 뒤 head를 갱신하세요.
        new_node = Node(data)
        new_node.link = self.head
        self.head = new_node
        self.size += 1

    def insert_after(self, prev: Node, data: Any) -> None:
        # TODO 5: 순서에 유의하세요. new_node -> 기존 다음, prev -> new_node
        if prev is None:
            raise ValueError("prev는 None일 수 없습니다.")

        new_node = Node(data)
        new_node.link = prev.link
        prev.link = new_node
        self.size += 1

    def insert_last(self, data: Any) -> None:
        # TODO 6: 빈 리스트/비어 있지 않은 리스트를 모두 처리하세요.
        new_node = Node(data)

        if self.is_empty():
            self.head = new_node
        else:
            current = self.head

            while current.link is not None:
                current = current.link

            current.link = new_node

        self.size += 1

    def delete_first(self) -> Any:
        # TODO 7: 비어 있으면 IndexError, 아니면 첫 노드를 제거·반환하세요.
        if self.is_empty():
            raise IndexError("빈 리스트에서는 삭제할 수 없습니다.")

        removed = self.head
        self.head = self.head.link
        self.size -= 1

        return removed.data

    def delete_after(self, prev: Node) -> Any:
        # TODO 8: prev 다음 노드를 제거·반환하세요.
        if prev is None or prev.link is None:
            raise IndexError("삭제할 다음 노드가 없습니다.")

        removed = prev.link
        prev.link = removed.link
        self.size -= 1

        return removed.data

    def search(self, target: Any) -> int:
        # TODO 9: target의 인덱스. 없으면 -1을 반환하세요.
        current = self.head
        index = 0

        while current is not None:
            if current.data == target:
                return index

            current = current.link
            index += 1

        return -1

    def get_node(self, i: int) -> Node:
        # TODO 10: i번째 Node를 반환하세요. 범위를 벗어나면 IndexError.
        if i < 0 or i >= self.size:
            raise IndexError("인덱스 범위를 벗어났습니다.")

        current = self.head

        for _ in range(i):
            current = current.link

        return current

    def get(self, i: int) -> Any:
        return self.get_node(i).data

    def reverse(self) -> None:
        # TODO 11 (도전): 새 노드 없이 link만 재연결해 뒤집으세요.
        prev = None
        current = self.head

        while current is not None:
            next_node = current.link

            current.link = prev

            prev = current
            current = next_node

        self.head = prev

    def middle(self) -> Any:
        # TODO 12 (도전): slow/fast 포인터로 한 번의 순회에서 중간 값을 반환하세요.
        if self.is_empty():
            raise IndexError("빈 리스트에는 중간 값이 없습니다.")

        slow = self.head
        fast = self.head

        while fast is not None and fast.link is not None:
            slow = slow.link
            fast = fast.link.link

        return slow.data

## 4. 단계별 테스트
아래 셀은 해당 TODO를 해결한 뒤 실행하세요.

In [3]:
# TODO 1~3 확인
ll = LinkedList()
assert ll.is_empty() is True
assert len(ll) == 0
assert ll.to_list() == []
print('기본 상태 통과')

기본 상태 통과


In [4]:
# TODO 4, 6 확인
ll.insert_first('B')
ll.insert_first('A')
ll.insert_last('C')
print(ll.to_list())
assert ll.to_list() == ['A', 'B', 'C']
assert len(ll) == 3
print('삽입 테스트 통과')

['A', 'B', 'C']
삽입 테스트 통과


In [5]:
# TODO 5 확인
prev = ll.get_node(1)  # 'B' 노드
ll.insert_after(prev, 'X')
print(ll.to_list())
assert ll.to_list() == ['A', 'B', 'X', 'C']
print(' 중간 삽입 테스트 통과')

['A', 'B', 'X', 'C']
 중간 삽입 테스트 통과


In [6]:
# TODO 7~10 확인
assert ll.delete_first() == 'A'
assert ll.delete_after(ll.get_node(0)) == 'X'
assert ll.to_list() == ['B', 'C']
assert ll.search('C') == 1
assert ll.search('Z') == -1
assert ll.get(0) == 'B'
print(' 삭제·탐색·접근 테스트 통과')

 삭제·탐색·접근 테스트 통과


---
## 5. 도전 과제

### A. `reverse()`
- `prev = None`, `cur = self.head`에서 시작합니다.
- 반복마다 다음 노드를 잠시 저장한 뒤, 현재 노드의 링크를 반대로 돌립니다.
- 마지막에는 `head`를 `prev`로 바꿉니다.

### B. `middle()`
- `slow`, `fast`를 모두 head에서 시작합니다.
- `slow`는 한 칸, `fast`는 두 칸 전진합니다.
- `fast`가 끝에 도달하면 `slow`가 중간에 있습니다.

In [7]:
# TODO 11~12 확인
challenge = LinkedList()
for value in [10, 20, 30, 40, 50]:
    challenge.insert_last(value)
assert challenge.middle() == 30
challenge.reverse()
print(challenge.to_list())
assert challenge.to_list() == [50, 40, 30, 20, 10]
print(' 도전 과제 통과')

[50, 40, 30, 20, 10]
 도전 과제 통과


---
## 제출 전 자기 점검
- [ ] 빈 리스트에서 삭제할 때 적절히 `IndexError`를 발생시키는가?
- [ ] `size`가 삽입/삭제마다 정확히 갱신되는가?
- [ ] `insert_after`에서 링크를 잃지 않는 순서를 지켰는가?
- [ ] `reverse`에서 새 노드를 만들지 않았는가?
- [ ] 배열 기반 리스트와 연결 리스트의 차이를 Big-O 기준으로 설명할 수 있는가?

## 생각해 보기 (답은 Markdown 셀에 작성)
1. `tail`을 저장하면 어떤 연산이 빨라지고, 삭제 시 어떤 추가 처리가 필요할까요?
2. 인덱스 접근이 잦고 중간 삽입/삭제가 드문 프로그램에는 배열과 연결 리스트 중 무엇이 더 적합할까요?
3. 연결 리스트의 이론적 장점이 실제 실행 시간에서 항상 우세하지 않은 이유를 캐시 지역성과 함께 설명해 보세요.

**답안 작성**

1. `tail`을 저장하면 리스트의 마지막 노드를 찾기 위해 처음부터 순회할 필요가 없으므로, 맨 뒤 삽입 연산을 O(n)에서 O(1)로 줄일 수 있다. 
   다만 마지막 노드를 삭제하거나 리스트가 비게 되는 경우에는 `tail`도 올바른 노드를 가리키도록 갱신해야 한다. 특히 단일 연결 리스트에서는 마지막 노드를 삭제할 때 이전 노드를 바로 알 수 없기 때문에, 이전 노드를 찾기 위한 순회가 필요할 수 있다.

2. 인덱스 접근이 자주 발생하고 중간 삽입/삭제가 드문 프로그램에는 배열이 더 적합하다.  
   배열은 인덱스를 이용하여 원하는 원소에 O(1) 시간에 접근할 수 있지만, 연결 리스트는 원하는 위치까지 노드를 따라가야 하므로 O(n)이 필요하다. 따라서 삽입/삭제보다 임의 접근이 중요한 경우에는 배열이 유리하다.

3. 연결 리스트는 노드의 위치를 알고 있다면 삽입/삭제 시 원소들을 이동할 필요가 없어 이론적으로 효율적이다. 그러나 실제 실행 시간에서는 항상 배열보다 빠른 것은 아니다.  
   배열은 원소들이 메모리의 연속된 공간에 저장되므로 CPU가 주변 데이터를 캐시에 함께 가져올 수 있어 캐시 지역성(cache locality)이 좋다. 반면 연결 리스트의 노드들은 메모리 곳곳에 흩어져 있을 수 있기 때문에 다음 노드로 이동할 때마다 새로운 메모리 영역을 읽어야 하고, 캐시 미스가 자주 발생할 수 있다. 또한 각 노드가 다음 노드의 주소를 저장해야 하므로 추가적인 메모리 비용도 발생한다. 따라서 이론적인 시간 복잡도가 유리하더라도 실제 실행에서는 배열이 더 빠른 경우가 많다.